In [3]:
import os
import json
import re

class Node:
    def __init__(self, operator=None, criteria=None):
        self.operator = operator
        self.criteria = criteria
        self.left = None
        self.right = None

    def to_dict(self):
        if self.operator:
            children = {}
            if self.left:
                children["left"] = self.left.to_dict()
            if self.right:
                children["right"] = self.right.to_dict()
            return {self.operator: children}
        else:
            return {"raw_text": self.criteria if self.criteria is not None else "empty set"}

def parse_text(text, operator):
    if not text:
        return Node(criteria="empty set")

    sentences = [s.strip() for s in text.split("\n") if s.strip()]
    if not sentences:
        return Node(criteria="empty set")

    root = Node(operator=operator)
    current_node = root

    for i, sentence in enumerate(sentences):
        if i == 0:
            current_node.left = Node(criteria=sentence)
        else:
            new_node = Node(operator=operator)
            current_node.right = new_node
            current_node = new_node
            current_node.left = Node(criteria=sentence)

    return root

def build_tree(inclusion_text, exclusion_text):
    inclusion_tree = parse_text(inclusion_text, "AND")
    exclusion_tree = parse_text(exclusion_text, "NOT OR")

    root = Node(operator="AND")
    root.left = inclusion_tree
    root.right = exclusion_tree

    return root

def read_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return f.read()

def save_json_to_file(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2)

def process_files(ann_dir, txt_dir, output_dir):
    ann_files = [f for f in os.listdir(ann_dir) if f.endswith('.ann')]
    txt_files = [f for f in os.listdir(txt_dir) if f.endswith('.txt')]

    for ann_file in ann_files:
        txt_file = ann_file.replace('.ann', '.txt')

        if txt_file not in txt_files:
            print(f"Missing corresponding .txt file for {ann_file}")
            continue

        ann_path = os.path.join(ann_dir, ann_file)
        txt_path = os.path.join(txt_dir, txt_file)

        txt_content = read_file(txt_path)

        inclusion_text = ""
        exclusion_text = ""

        lines = txt_content.split("\n")
        inclusion_started = False
        exclusion_started = False

        for line in lines:
            if "Inclusion Criteria" in line:
                inclusion_started = True
                exclusion_started = False
                continue
            elif "Exclusion Criteria" in line:
                exclusion_started = True
                inclusion_started = False
                continue

            if inclusion_started:
                inclusion_text += line.strip() + " "
            elif exclusion_started:
                exclusion_text += line.strip() + " "

        inclusion_text = inclusion_text.strip()
        exclusion_text = exclusion_text.strip()

        tree = build_tree(inclusion_text, exclusion_text)
        tree_json = tree.to_dict()
        output_path = os.path.join(output_dir, ann_file.replace('.ann', '_parsed.json'))
        save_json_to_file(tree_json, output_path)
        print(f"Processed {ann_file} and saved to {output_path}")

In [4]:
ann_directory = 'data'
txt_directory = 'data'
output_directory = 'output'

if not os.path.exists(output_directory):
    os.makedirs(output_directory)

process_files(ann_directory, txt_directory, output_directory)

Processed NCT03860012.ann and saved to output\NCT03860012_parsed.json
Processed NCT03860025.ann and saved to output\NCT03860025_parsed.json
Processed NCT03860038.ann and saved to output\NCT03860038_parsed.json
Processed NCT03860064.ann and saved to output\NCT03860064_parsed.json
Processed NCT03860090.ann and saved to output\NCT03860090_parsed.json
Processed NCT03860103.ann and saved to output\NCT03860103_parsed.json
Processed NCT03860116.ann and saved to output\NCT03860116_parsed.json
Processed NCT03860142.ann and saved to output\NCT03860142_parsed.json
Processed NCT03860168.ann and saved to output\NCT03860168_parsed.json
Processed NCT03860181.ann and saved to output\NCT03860181_parsed.json
Processed NCT03860194.ann and saved to output\NCT03860194_parsed.json
Processed NCT03860220.ann and saved to output\NCT03860220_parsed.json
Processed NCT03860233.ann and saved to output\NCT03860233_parsed.json
Processed NCT03860246.ann and saved to output\NCT03860246_parsed.json
Processed NCT0386025